# TCGA → embedding H-optimus-0 (runtime GPU)

Seconda fase dell'estrazione TCGA (docs/mvp.md): `td task=stream` sui vetrini che `tcga_fetch_colab.ipynb` ha messo su Drive. Per ogni vetrino: copia sul disco locale della VM (`stream.stage_dir`, openslide legge a salti e su Drive sarebbe lento), maschera del tessuto, QC e griglia, embedding sulla GPU, salvataggio su Drive, poi cancella il vetrino da Drive (`stream.delete_raw`).

- **Riprendibile.** Basta rieseguire le celle: i vetrini già embeddati si saltano.
- **Non scarica nulla** se il vetrino è già su Drive; quelli mancanti li scarica da GDC (più lento).
- **Tempo.** Stima 10–20 mila tile a vetrino (4 vetrini di prova sul Mac: 3.949–19.396), ~1,5–3 min su A100: circa 5 h per lotto.
- **Spazio.** ~9 GB di embedding per lotto, più tile e QC.

**Prima di iniziare:** Runtime → Cambia tipo di runtime → **GPU A100** (o L4). Secret `HF_TOKEN` con accesso a https://huggingface.co/bioptimus/H-optimus-0.

In [ ]:
# 0. Hardware: GPU, CPU e disco locale (i vetrini passano di qui uno alla volta, più il prefetch)
!nvidia-smi -L
!nproc
!df -h /content | tail -1

In [ ]:
# 1. Google Drive: vetrini, embedding, tile e QC finiscono qui, così sopravvivono alle disconnessioni
import shutil
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/tumordetector/tcga")      # come data/tcga sul Mac
RAW = Path("/content/drive/MyDrive/tumordetector/raw/tcga")  # i vetrini scaricati da td task=fetch
OUT.mkdir(parents=True, exist_ok=True)
RAW.mkdir(parents=True, exist_ok=True)
print(f"spazio libero su Drive: {shutil.disk_usage(OUT).free / 1e9:.0f} GB")

In [ ]:
# 2. Codice (se il repository è privato, clonalo con un token o rendilo pubblico)
REPO_URL = "https://github.com/masnarivan/TumorDetector.git"
REV = "origin/main"    # o un commit preciso; origin/ perché alla seconda sessione il clone esiste già e va aggiornato
!git clone -q {REPO_URL} /content/TumorDetector 2>/dev/null || git -C /content/TumorDetector fetch -q
!git -C /content/TumorDetector checkout -q {REV}
# Il progetto dichiara Python 3.12; Colab può essere più avanti (3.13 a ottobre 2026)
!pip install -q --ignore-requires-python -e /content/TumorDetector
# Le librerie che decidono maschera, griglia e pixel delle tile alle versioni di uv.lock: con quelle
# preinstallate su Colab la griglia può cambiare (e scikit-image < 0.26 non ha max_size)
!pip install -q scikit-image==0.26.0 scipy==1.18.1 pillow==12.3.0 shapely==2.2.0 \
    openslide-python==1.4.6 openslide-bin==4.0.1.2 timm==1.0.30
%cd /content/TumorDetector

In [ ]:
# 3. Pesi di H-optimus-0 (modello ad accesso controllato), con il token dal secret HF_TOKEN
import os
from huggingface_hub import snapshot_download
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print("secret HF_TOKEN non disponibile:", type(e).__name__)
    from huggingface_hub import notebook_login
    notebook_login()
snapshot_download("bioptimus/H-optimus-0")

In [ ]:
# 4. Prova: i 4 vetrini già tilati sul Mac. La cella 6 confronta la griglia con quella del Mac
#    (notebooks/tcga_tile_counts.csv): deve essere identica prima di lanciare il lotto.
import pandas as pd
TRIAL = list(pd.read_csv("notebooks/tcga_tile_counts.csv").slide_id)
BATCH_SIZE, THREADS, PREFETCH = 256, 8, 2   # velocità, non cambiano gli embedding (T4: 64 / 4)
SLIDES = "[" + ",".join(TRIAL) + "]"
!td task=stream backbone=h_optimus_0 data=tcga data.raw_dir={RAW} paths.out={OUT} "data.slides={SLIDES}" \
    stream.stage_dir=/content/stage stream.delete_raw=false stream.prefetch={PREFETCH} \
    embed.batch_size={BATCH_SIZE} embed.threads={THREADS} mlflow.tracking_uri=sqlite:///{OUT}/mlflow.db

In [ ]:
# 5. Un lotto intero (1 o 2), solo i vetrini già su Drive; cancellati da Drive dopo l'embedding
BATCH = 1
m = pd.read_csv("data/manifests/tcga.csv")
ids = [s for s in m[m.batch == BATCH].slide_id if (RAW / f"{s}.svs").exists()]
print(len(ids), "vetrini del lotto", BATCH, "su Drive")
SLIDES = "[" + ",".join(ids) + "]"
!td task=stream backbone=h_optimus_0 data=tcga data.raw_dir={RAW} paths.out={OUT} "data.slides={SLIDES}" \
    stream.stage_dir=/content/stage stream.delete_raw=true stream.prefetch={PREFETCH} \
    embed.batch_size={BATCH_SIZE} embed.threads={THREADS} mlflow.tracking_uri=sqlite:///{OUT}/mlflow.db

In [ ]:
# 6. Stato (eseguibile in qualsiasi momento, anche da una nuova sessione)
root = OUT / "embeddings" / "h_optimus_0"
m = pd.read_csv("data/manifests/tcga.csv")
for b, g in m.groupby("batch"):
    print(f"lotto {b}: {sum((root / f'{s}.npy').exists() for s in g.slide_id)} / {len(g)} vetrini con embedding")
if (root / "extract_log.csv").exists():
    log = pd.read_csv(root / "extract_log.csv")   # ultima esecuzione
    done = log[log.status == "embedded"]
    if len(done):
        wall = done.t_wait_s.sum() + done.t_tile_s.sum() + done.t_embed_s.sum()
        print(f"ultima esecuzione: {len(done)} vetrini, {done.n_tiles.sum():,.0f} tile, "
              f"{done.n_tiles.sum() / max(done.t_embed_s.sum(), 1):.0f} tile/s di embedding, "
              f"{wall / len(done) / 60:.1f} min per vetrino")
    mac = pd.read_csv("notebooks/tcga_tile_counts.csv").set_index("slide_id").n_tiles
    grid = done.set_index("slide_id").n_tiles.reindex(mac.index).dropna()
    if len(grid):
        diff = grid[grid != mac.reindex(grid.index)]
        print("griglia uguale a quella del Mac" if diff.empty else f"ATTENZIONE, griglia diversa dal Mac: {diff.to_dict()}")
    bad = log[(log.status == "error") | log.status.str.startswith("referred")]
    if len(bad):
        display(bad)
print(f"spazio libero su Drive: {shutil.disk_usage(OUT).free / 1e9:.0f} GB")

Alla fine, sul Mac: copia `tumordetector/tcga` da Drive in `data/tcga` (embedding, tile e QC), poi `td task=verify data=tcga backbone=h_optimus_0` e `td task=index data=tcga backbone=h_optimus_0`.